In [1]:
import os, sys
os.chdir("/files/week-03/Project/code")
sys.path.insert(0, os.getcwd())
print(os.getcwd(), os.path.exists("helpers.py"))

/files/week-03/Project/code True


In [2]:
from helpers import *
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
import pandas_datareader.data as web

RES = Path("../results")

macro = pd.read_csv(DATA / "macro_monthly.csv", index_col=0, parse_dates=True)
factors = pd.read_csv(DATA / "factors_monthly.csv", index_col=0, parse_dates=True)

common = macro.join(factors, how="inner")
macro_c = common[macro.columns]
train_m = macro_c[macro_c.index <= SPLIT]
test_m = macro_c[macro_c.index > SPLIT]
print(train_m.shape, test_m.shape)

(438, 5) (320, 5)


In [3]:
scaler = StandardScaler().fit(train_m)
X_train = scaler.transform(train_m)
X_test = scaler.transform(test_m)

pca = PCA(n_components=3).fit(X_train)
Z_train = pca.transform(X_train)
Z_test = pca.transform(X_test)

cols = ["PC1", "PC2", "PC3"]
pca_var = pd.Series(pca.explained_variance_ratio_, index=cols, name="variance_expliquee").round(3)
pca_poids = pd.DataFrame(pca.components_.T, index=macro.columns, columns=cols).round(2)

pca_var.to_csv(RES / "pca_variance.csv")
pca_poids.to_csv(RES / "pca_poids.csv")
print(pca_var)
print(pca_poids)

PC1    0.496
PC2    0.293
PC3    0.108
Name: variance_expliquee, dtype: float64
                PC1   PC2   PC3
slope          0.34 -0.64  0.18
credit_spread  0.56 -0.06 -0.30
unemployment   0.55 -0.13  0.48
inflation      0.23  0.68  0.53
indpro_growth -0.47 -0.34  0.60


In [4]:
bic = {}
for k in range(2, 7):
    g = GaussianMixture(n_components=k, covariance_type="full", n_init=10, random_state=42).fit(Z_train)
    bic[k] = round(g.bic(Z_train), 1)

bic = pd.Series(bic, name="BIC")
bic.index.name = "k"
bic.to_csv(RES / "gmm_bic.csv")
print(bic)

k
2    3791.0
3    3671.6
4    3640.3
5    3505.9
6    3475.9
Name: BIC, dtype: float64


In [5]:
for k in [3, 4]:
    g = GaussianMixture(n_components=k, covariance_type="full", n_init=10, random_state=42).fit(Z_train)
    labels = g.predict(Z_train)

    profil = train_m.groupby(labels).mean().round(2)
    profil.insert(0, "mois", pd.Series(labels).value_counts().sort_index().values)
    profil = profil.join(common.loc[train_m.index, FACT].groupby(labels).mean().round(2))
    profil.index.name = "regime"

    profil.to_csv(RES / f"profil_regimes_k{k}.csv")
    print("k =", k)
    print(profil)

k = 3
        mois  slope  credit_spread  unemployment  inflation  indpro_growth  \
regime                                                                       
0        211   0.86           1.47          4.84       3.41           4.56   
1        125   2.74           2.05          7.00       3.61           4.76   
2        102   0.78           2.05          7.26       9.18          -0.09   

        size  value  quality  momentum  
regime                                  
0      -0.09   0.10     0.15      0.93  
1       0.13   0.69     0.42      0.66  
2       0.86   0.45    -0.07      1.10  
k = 4
        mois  slope  credit_spread  unemployment  inflation  indpro_growth  \
regime                                                                       
0        123   2.74           2.06          6.99       3.55           4.69   
1        208   0.88           1.47          4.84       3.38           4.49   
2         47   1.78           2.78          8.66       8.62          -4.85   
3 

In [6]:
gmm4 = GaussianMixture(n_components=4, covariance_type="full", n_init=10, random_state=42).fit(Z_train)
reg_train = pd.Series(gmm4.predict(Z_train), index=train_m.index)
reg_test = pd.Series(gmm4.predict(Z_test), index=test_m.index)

# Numérotation stable : 0 expansion, 1 surchauffe, 2 récession, 3 reprise
prof = train_m.groupby(reg_train).mean()
rec = prof["indpro_growth"].idxmin()
reste = [r for r in prof.index if r != rec]
sur = prof.loc[reste, "inflation"].idxmax()
reste = [r for r in reste if r != sur]
exp = prof.loc[reste, "unemployment"].idxmin()
rep = [r for r in reste if r != exp][0]
mapping = {exp: 0, sur: 1, rec: 2, rep: 3}
reg_train = reg_train.map(mapping)
reg_test = reg_test.map(mapping)

blocs = (reg_train != reg_train.shift()).cumsum()
durees = reg_train.groupby(blocs).agg(["first", "size"])
durees.columns = ["regime", "mois"]
durees["debut"] = reg_train.index.to_series().groupby(blocs).first().dt.date
durees.to_csv(RES / "durees_regimes_train.csv")
print(len(durees), "blocs pour", len(reg_train), "mois")

reg_test.value_counts().sort_index().to_csv(RES / "regimes_test_comptes.csv")
print(reg_test["2008-09":"2009-06"].values)
print(reg_test["2020-03":"2020-12"].values)

profil_final = train_m.groupby(reg_train).mean().round(2)
profil_final.insert(0, "mois", reg_train.value_counts().sort_index().values)
profil_final = profil_final.join(common.loc[train_m.index, FACT].groupby(reg_train).mean().round(2))
profil_final.index.name = "regime"
profil_final.to_csv(RES / "profil_regimes_final.csv")
print(profil_final)

regimes = pd.concat([reg_train, reg_test]).rename("regime")
regimes.to_csv(DATA / "regimes_monthly.csv")
print(regimes.shape)
print(regimes.value_counts().sort_index())

16 blocs pour 438 mois
[0 0 0 0 0 0 0 2 2 0]
[3 2 2 0 0 0 0 0 0 0]
        mois  slope  credit_spread  unemployment  inflation  indpro_growth  \
regime                                                                       
0        208   0.88           1.47          4.84       3.38           4.49   
1         60   0.01           1.42          6.04       9.36           4.39   
2         47   1.78           2.78          8.66       8.62          -4.85   
3        123   2.74           2.06          6.99       3.55           4.69   

        size  value  quality  momentum  
regime                                  
0      -0.09   0.12     0.14      0.95  
1       0.62   0.64     0.01      1.30  
2       1.12   0.08    -0.10      0.81  
3       0.11   0.70     0.41      0.64  
(758,)
regime
0    392
1     63
2     65
3    238
Name: count, dtype: int64


In [8]:
usrec_path = DATA / "usrec_monthly.csv"
if not usrec_path.exists():
    web.DataReader("USREC", "fred", start="1963-01-01").to_csv(usrec_path)
usrec = pd.read_csv(usrec_path, index_col=0, parse_dates=True)["USREC"].reindex(regimes.index)

nber_n = pd.crosstab(regimes, usrec, rownames=["regime"], colnames=["NBER"])
nber_p = pd.crosstab(regimes, usrec, normalize="columns", rownames=["regime"], colnames=["NBER"]).round(2)
nber_n.to_csv(RES / "nber_comptes.csv")
nber_p.to_csv(RES / "nber_proportions.csv")
print(nber_n)
print(nber_p)

NBER      0   1
regime         
0       356  36
1        49  14
2        38  27
3       230   8
NBER       0     1
regime            
0       0.53  0.42
1       0.07  0.16
2       0.06  0.32
3       0.34  0.09
